In [1]:
# notebooks/11_prepare_fdi_data.ipynb

import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# -------------------------------------------------------------------------
# Step 1: Load and Sanitize Dataset B
# -------------------------------------------------------------------------
dataset_path = Path(
    '../data/fdi_Detection/Big_Datasets_for_Intelligent_Anomaly_Detection_Mi/'
    '2(c) current and voltage_timeseries_big_all_with_anomalies.csv'
)
assert dataset_path.exists(), f"Dataset file not found at: {dataset_path}"

df_raw = pd.read_csv(dataset_path)

# Sanitize headers (remove '#' and leading/trailing spaces)
df = df_raw.copy()
df.columns = df.columns.str.replace('#', '', regex=False).str.strip()

print(f"Total Rows Loaded: {len(df):,}")
print("Cleaned Columns:", list(df.columns))

# -------------------------------------------------------------------------
# Step 2: Formulate Multi-Feeder 12-Channel Grid & Binary Label
# -------------------------------------------------------------------------
# Create unified binary attack label at timestep level: 1 if any anomaly, else 0
df['is_attack'] = (
    (df['is_anomaly_voltage'].astype(bool)) |
    (df['is_anomaly_current'].astype(bool)) |
    (df['is_anomaly_frequency'].astype(bool))
).astype(int)

# Identify unique feeders and sort by timestamp
feeders = sorted(df['feeder'].unique())
print(f"\nUnique Feeders ({len(feeders)}): {feeders}")

# Pivot each feeder's 3 telemetry channels side-by-side indexed by time_s
telemetry_features = ['voltage_kV', 'current_A', 'frequency_Hz']

pivoted_dfs = []
for feeder_name in feeders:
    sub = df[df['feeder'] == feeder_name].sort_values('time_s').reset_index(drop=True)
    # Rename sensor columns to include feeder identity
    sub_features = sub[telemetry_features].rename(
        columns={c: f"{feeder_name}_{c}" for c in telemetry_features}
    )
    pivoted_dfs.append(sub_features)

# Combine 4 feeders side-by-side -> 12 continuous channels
X_time_series = pd.concat(pivoted_dfs, axis=1)

# Ground truth label: if ANY feeder has an attack at time t, grid label = 1
# Group by time_s and compute max of 'is_attack' across feeders
y_time_series = (
    df.groupby('time_s')['is_attack']
    .max()
    .reset_index(drop=True)
    .values
)

print(f"\nConstructed Multivariate Telemetry Array Shape: {X_time_series.shape} (7201 steps, 12 channels)")
print(f"Total Injection Attack Timestamps: {y_time_series.sum()} / {len(y_time_series)}")

# -------------------------------------------------------------------------
# Step 3: Chronological Train / Val / Test Split (Strictly Pre-Windowing)
# -------------------------------------------------------------------------
T_total = len(X_time_series)
idx_train = int(0.70 * T_total)  # 5,040
idx_val = int(0.85 * T_total)    # 6,120

X_train_raw = X_time_series.iloc[:idx_train].values
y_train_raw = y_time_series[:idx_train]

X_val_raw = X_time_series.iloc[idx_train:idx_val].values
y_val_raw = y_time_series[idx_train:idx_val]

X_test_raw = X_time_series.iloc[idx_val:].values
y_test_raw = y_time_series[idx_val:]

print(f"\nSplit Sizes (Continuous Time Steps):")
print(f"  • Train: {X_train_raw.shape[0]} intervals")
print(f"  • Val:   {X_val_raw.shape[0]} intervals")
print(f"  • Test:  {X_test_raw.shape[0]} intervals")

# -------------------------------------------------------------------------
# Step 4: Leakage-Free MinMax Normalization
# -------------------------------------------------------------------------
scaler = MinMaxScaler(feature_range=(0, 1))

# Fit ONLY on the training split
X_train_norm = scaler.fit_transform(X_train_raw)
# Transform val and test using fixed training parameters
X_val_norm = scaler.transform(X_val_raw)
X_test_norm = scaler.transform(X_test_raw)

# -------------------------------------------------------------------------
# Step 5: Sliding-Window Construction (Window = 30, Stride = 1)
# -------------------------------------------------------------------------
WINDOW_SIZE = 30  # 30 intervals * 0.5s = 15 seconds
STRIDE = 1

def create_sliding_windows(X: np.ndarray, y: np.ndarray, window_size: int = 30, stride: int = 1):
    X_windows, y_windows = [], []
    num_samples = len(X)
    for start in range(0, num_samples - window_size + 1, stride):
        end = start + window_size
        X_windows.append(X[start:end])
        # Window label is 1 if any timestep in the window has an attack
        y_windows.append(np.max(y[start:end]))
    return np.array(X_windows, dtype=np.float32), np.array(y_windows, dtype=np.int64)

X_train_win, y_train_win = create_sliding_windows(X_train_norm, y_train_raw, WINDOW_SIZE, STRIDE)
X_val_win, y_val_win = create_sliding_windows(X_val_norm, y_val_raw, WINDOW_SIZE, STRIDE)
X_test_win, y_test_win = create_sliding_windows(X_test_norm, y_test_raw, WINDOW_SIZE, STRIDE)

print(f"\nConstructed Sliding-Window Tensors:")
print(f"  • X_train: {X_train_win.shape} | Attack Windows: {y_train_win.sum()} ({y_train_win.mean()*100:.2f}%)")
print(f"  • X_val:   {X_val_win.shape}   | Attack Windows: {y_val_win.sum()} ({y_val_win.mean()*100:.2f}%)")
print(f"  • X_test:  {X_test_win.shape}  | Attack Windows: {y_test_win.sum()} ({y_test_win.mean()*100:.2f}%)")

# -------------------------------------------------------------------------
# Step 6: Save Artifacts for Model Fine-Tuning
# -------------------------------------------------------------------------
save_dir = Path('../data/processed_downstream')
save_dir.mkdir(parents=True, exist_ok=True)
save_file = save_dir / 'fdi_feeder_windows.npz'

np.savez_compressed(
    save_file,
    X_train=X_train_win, y_train=y_train_win,
    X_val=X_val_win,     y_val=y_val_win,
    X_test=X_test_win,   y_test=y_test_win,
    feature_names=list(X_time_series.columns)
)

print(f"\n[SUCCESS] Downstream FDI Dataset saved to: {save_file}")
print(f"File Size: {save_file.stat().st_size / (1024 * 1024):.2f} MB")

Total Rows Loaded: 28,804
Cleaned Columns: ['time_s', 'feeder', 'voltage_kV', 'current_A', 'frequency_Hz', 'is_anomaly_voltage', 'is_anomaly_current', 'is_anomaly_frequency']

Unique Feeders (4): ['12 kV Feeder', '3 kV Feeder', '5 kV Feeder', '8 kV Feeder']

Constructed Multivariate Telemetry Array Shape: (7201, 12) (7201 steps, 12 channels)
Total Injection Attack Timestamps: 75 / 7201

Split Sizes (Continuous Time Steps):
  • Train: 5040 intervals
  • Val:   1080 intervals
  • Test:  1081 intervals

Constructed Sliding-Window Tensors:
  • X_train: (5011, 30, 12) | Attack Windows: 1581 (31.55%)
  • X_val:   (1051, 30, 12)   | Attack Windows: 262 (24.93%)
  • X_test:  (1052, 30, 12)  | Attack Windows: 210 (19.96%)

[SUCCESS] Downstream FDI Dataset saved to: ..\data\processed_downstream\fdi_feeder_windows.npz
File Size: 0.37 MB
